In [59]:
# !pip install tensorly
# !pip install tensorly-torch

In [60]:
import torch
import torch.nn as nn
class reshape(nn.Module):
    '''
    reshapes the 3-order tensor into 6-order tensor

    ----------
    split : list
        split indices to be applied to each mode of the 3-order tensor

    map_type : int
        based on attached - 1 or compressed - 2 splitting method

    device : str
        operation device, default value is cpu


    inputs a 3-order torch.tensor

    returns a 6-order torch.tensor
    '''
    def __init__(self, split, map_type=1, device='cpu'):
        super(reshape, self).__init__()

        self.split = split
        self.map_type = map_type
        self.device = device

    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []

        if self.map_type == 1:
            # Approach 1 : Attached
            C_indices, H_indices, W_indices = [
                [sum(dim // self.split[i] for _ in range(j)) for j in range(self.split[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]

            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1],
                                           H_indices[j]:H_indices[j+1],
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            # Approach 2 : Compressed
            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            C_stride_indices = torch.arange(i, C, self.split[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.split[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.split[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)

        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.split[0] * self.split[1] * self.split[2])
        result = torch.cat(chunks).view(
            batch_size, self.split[0], self.split[1], self.split[2],
            *chunks[0].shape[1:])
        return result

    def forward(self, x):
        chunks = self.split_into_chunks(x)
        output = self.stack_chunks_to_form_tensor(chunks)
        return output

In [61]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

In [62]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [63]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 8

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [64]:
def topk_accuracy(outputs, targets, topk=(1,)):
    '''
    calculates top-k accuracy

    ----------
    outputs : torch.tensor

    targets : torch.tensor

    topk  : tuple
        calculates top k accuracy given outpurs and targets


    reutrns a python dictionary of top i <= k accuracies

    '''
    maxk = max(topk)
    _, topk_indices = torch.topk(input=outputs, k=maxk, dim=1, largest=True, sorted=True)
    correct = topk_indices.eq(targets.view(-1, 1).expand_as(topk_indices))
    accuracies = {}
    for k in topk:
        correct_k = correct[:,:k].float().sum()
        accuracies[k] = {'correct':correct_k, 'accuracy': (correct_k / outputs.shape[0]) * 100.0}
    return accuracies

In [65]:
def cp(module):
  return sum(p.numel() for p in module.parameters())

In [66]:
def print_gpu_memory_usage(stage):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    string = f'{stage} - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB'
    print(string)
    return string


In [67]:
def append_to_file(file_name, text):
    with open(file_name, 'a') as file:
        file.write(text + '\n')

# FC layers

In [68]:
class CNN1(nn.Module):
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256, bias = False)
        self.fc2 = nn.Linear(256,10, bias = False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model1 = CNN1().to(device)


In [69]:
classifier1 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256, bias = False),
    nn.Linear(256, 10, bias = False)
)

print(cp(classifier1))
append_to_file(file_name='TCL&TRL_report.txt', text=f'FC classifier # parameters {cp(classifier1)}')

1051136


In [70]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters())

In [71]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model1.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model1(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model1.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [72]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'FC took {end_time - start_time} time')
append_to_file(file_name='TCL&TRL_report.txt', text=f'FC had {string}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'FCL last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

total forward time : 1.1183180809020996, total backward time : 1.2610132694244385
Train epoch 1: top1=0.5719599723815918%, top2=0.7624399662017822%, top3=0.8549399971961975%, top4=0.9128599762916565%, top5=0.9451799988746643%, loss=0.1498881022375822, time=7.650006532669067s
Test epoch 1: top1=0.6491000056266785%, top2=0.8277999758720398%, top3=0.9017999768257141%, top4=0.9435999989509583%, top5=0.9660999774932861%, loss=0.12380708733201026, time=0.9908971786499023s
Memory Usage  - Allocated: 32.97 MB, Reserved: 70.00 MB
total forward time : 1.1166632175445557, total backward time : 1.1858580112457275
Train epoch 2: top1=0.7021399736404419%, top2=0.8597599864006042%, top3=0.9230999946594238%, top4=0.9584999680519104%, top5=0.9774199724197388%, loss=0.10602214690037072, time=7.5486366748809814s
Test epoch 2: top1=0.6865999698638916%, top2=0.8547999858856201%, top3=0.9214999675750732%, top4=0.9578999876976013%, top5=0.977400004863739%, loss=0.11103267110139131, ti

In [73]:
append_to_file(file_name='TCL&TRL_report.txt', text=f'########################################')

# TCL & TRL from Tensorly

In [74]:
class CNN2(nn.Module):
    def __init__(self):
        super(CNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL(input_shape = (64,8,8), rank = (64,2,2))
        self.trl = TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = self.trl(x)
        return x


model2 = CNN2().to(device)


In [75]:
classifier2 = nn.Sequential(
    TCL(input_shape = (64,8,8), rank = (64,2,2)),
    TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
)

print(cp(classifier2))
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly classifier # parameters {cp(classifier2)}')

4972


In [76]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters())

In [77]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model2.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model2(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
    

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model2.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [78]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly took {end_time - start_time} time')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly had {string}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL tensorly last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

total forward time : 2.311960458755493, total backward time : 2.2182769775390625
Train epoch 1: top1=0.460999995470047%, top2=0.6719799637794495%, top3=0.7840399742126465%, top4=0.8598799705505371%, top5=0.9102999567985535%, loss=0.18367038785457612, time=9.634662389755249s
Test epoch 1: top1=0.5582000017166138%, top2=0.7591999769210815%, top3=0.8556999564170837%, top4=0.920199990272522%, top5=0.9547999501228333%, loss=0.15371360208392143, time=1.2222023010253906s
Memory Usage  - Allocated: 24.99 MB, Reserved: 70.00 MB
total forward time : 2.2886879444122314, total backward time : 2.304265260696411
Train epoch 2: top1=0.5971199870109558%, top2=0.7915399670600891%, top3=0.8782199621200562%, top4=0.9295799732208252%, top5=0.9602399468421936%, loss=0.140644982483387, time=9.595869779586792s
Test epoch 2: top1=0.583299994468689%, top2=0.7749999761581421%, top3=0.8642999529838562%, top4=0.920199990272522%, top5=0.9532999992370605%, loss=0.1446560114249587, time=1.280

In [79]:
append_to_file(file_name='TCL&TRL_report.txt', text=f'########################################')

# TCL & TRL Method 1 just 3D tensors

In [80]:
class TCL2(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL2, self).__init__()
          #suppose it is 3d :
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)

    def forward(self, x):
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')

          return x

In [81]:
import math
class TRL2(nn.Module):
    def __init__(self, input_shape, output_shape, rank, bias = False): # last rank for the output shape
          super(TRL2, self).__init__()
          self.G = nn.Parameter(torch.empty(rank), requires_grad=True)
          nn.init.kaiming_uniform_(self.G, a=math.sqrt(5))
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)
          self.fc4 = nn.Linear(rank[3], output_shape[0], bias = bias)

    def forward(self, x):
          G = self.fc4(self.G)
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')
          x = torch.einsum('b x y z , x y z d -> b d', x, G)

          return x

In [82]:
class CNN3(nn.Module):
    def __init__(self):
        super(CNN3, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL2(input_shape = (64,8,8), rank = (64,4,4))
        self.trl = TRL2(input_shape = (64,4,4), output_shape = (10,), rank=(10,1,1,10))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = self.trl(x)
        return x


model3 = CNN3().to(device)


In [83]:
classifier3 = nn.Sequential(
    TCL2(input_shape = (64,8,8), rank = (64,2,2)),
    TRL2(input_shape = (64,2,2), output_shape = (10, ), rank=(10,1,1,10))
)

print(cp(classifier3))
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 classifier # parameters {cp(classifier3)}')

4972


In [84]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters())

In [85]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model3.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model3(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
        

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model3.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [86]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 took {end_time - start_time} time')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 had {string}')
append_to_file(file_name='TCL&TRL_report.txt', text=f'TCL & TRL method 1 last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

total forward time : 2.0347955226898193, total backward time : 2.174055337905884
Train epoch 1: top1=0.41547998785972595%, top2=0.6225799918174744%, top3=0.7439199686050415%, top4=0.8295799493789673%, top5=0.8883399963378906%, loss=0.19797963260412216, time=9.357771158218384s
Test epoch 1: top1=0.5238000154495239%, top2=0.7295999526977539%, top3=0.8382999897003174%, top4=0.9070999622344971%, top5=0.9436999559402466%, loss=0.1613968932569027, time=1.1910724639892578s
Memory Usage  - Allocated: 24.99 MB, Reserved: 70.00 MB
total forward time : 1.9908251762390137, total backward time : 2.108389377593994
Train epoch 2: top1=0.5668399930000305%, top2=0.7664999961853027%, top3=0.8598999977111816%, top4=0.9167999625205994%, top5=0.9519599676132202%, loss=0.14984116951704024, time=9.10598111152649s
Test epoch 2: top1=0.6074999570846558%, top2=0.7945999503135681%, top3=0.8787999749183655%, top4=0.9267999529838562%, top5=0.9587999582290649%, loss=0.13912094271332026, time

In [87]:
append_to_file(file_name='TCL&TRL_report.txt', text=f'########################################')